Implement error handling for invalid inputs by designing a robust validation framework that detects and corrects potential errors, guiding users with contextual error messages and intelligent input suggestions.

In [1]:
import pandas as pd
import numpy as np
import joblib
import ipywidgets as widgets

from IPython.display import display, clear_output

In [2]:
df = pd.read_csv("heart_disease_train.csv")

model = joblib.load("saved_models/heart_disease_model_v1.joblib")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

print("Dataset loaded:", df.shape)
print("Model loaded successfully!")

Dataset loaded: (79, 19)
Model loaded successfully!


In [ ]:
validation_rules = {
    "age": (1, 120),
    "trestbps": (50, 250),
    "chol": (50, 700),
    "thalach": (40, 250),
    "oldpeak": (0, 10)
}

for feature in features:
    print(
        feature,
        "Minimum:", validation_rules[feature][0],
        "Maximum:", validation_rules[feature][1]
    ) 

age Minimum: 1 Maximum: 120
trestbps Minimum: 50 Maximum: 250
chol Minimum: 50 Maximum: 700
thalach Minimum: 40 Maximum: 250
oldpeak Minimum: 0 Maximum: 10


In [4]:
def validate_input(user_data):

    cleaned_data = {}
    errors = {}
    suggestions = {}

    for feature in features:

        value = user_data.get(feature)

        if value is None or str(value).strip() == "":
            errors[feature] = "This field is required."
            suggestions[feature] = f"Try {df[feature].median():.2f}"
            continue

        try:
            value = float(str(value).strip())

            if not np.isfinite(value):
                raise ValueError

        except (ValueError, TypeError):
            errors[feature] = "Please enter a valid numeric value."
            suggestions[feature] = f"Example: {df[feature].median():.2f}"
            continue

        minimum, maximum = validation_rules[feature]

        if value < minimum or value > maximum:
            errors[feature] = f"Value must be between {minimum} and {maximum}."
            suggestions[feature] = (
                f"Suggested dataset range: "
                f"{df[feature].min()} to {df[feature].max()}"
            )
            continue

        cleaned_data[feature] = value

    if errors:
        return None, errors, suggestions

    return pd.DataFrame([cleaned_data], columns=features), {}, {}

In [5]:
test_cases = [
    {
        "age": "45",
        "trestbps": "120",
        "chol": "200",
        "thalach": "150",
        "oldpeak": "1.2"
    },
    {
        "age": "",
        "trestbps": "120",
        "chol": "200",
        "thalach": "150",
        "oldpeak": "1.2"
    },
    {
        "age": "abc",
        "trestbps": "120",
        "chol": "200",
        "thalach": "150",
        "oldpeak": "1.2"
    },
    {
        "age": "150",
        "trestbps": "120",
        "chol": "200",
        "thalach": "150",
        "oldpeak": "1.2"
    },
    {
        "age": "45",
        "trestbps": "inf",
        "chol": "200",
        "thalach": "150",
        "oldpeak": "1.2"
    }
]

for i, test in enumerate(test_cases, 1):

    result, errors, suggestions = validate_input(test)

    print(f"\nTest Case {i}")

    if errors:
        for feature, error in errors.items():
            print("Error:", feature, "-", error)
            print("Suggestion:", suggestions[feature])
    else:
        print("All inputs are valid.")


Test Case 1
All inputs are valid.

Test Case 2
Error: age - This field is required.
Suggestion: Try 56.00

Test Case 3
Error: age - Please enter a valid numeric value.
Suggestion: Example: 56.00

Test Case 4
Error: age - Value must be between 1 and 120.
Suggestion: Suggested dataset range: 29 to 77

Test Case 5
Error: trestbps - Please enter a valid numeric value.
Suggestion: Example: 154.00


In [6]:
input_widgets = {}

for feature in features:

    minimum, maximum = validation_rules[feature]

    input_widgets[feature] = widgets.Text(
        value=str(df[feature].median()),
        description=feature,
        placeholder=f"Enter {feature}",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="90%")
    )

display(widgets.VBox(list(input_widgets.values())))

In [ ]:
output = widgets.Output()

def check_inputs(button):

    with output:
        clear_output(wait=True)

        user_data = {
            feature: input_widgets[feature].value
            for feature in features
        }

        result, errors, suggestions = validate_input(user_data)

        if errors:

            print("Please correct the following inputs:\n")

            for feature, error in errors.items():
                print(f"❌ {feature}: {error}")
                print(f"💡 Suggestion: {suggestions[feature]}\n")

        else:
            print("All inputs are valid!")
            display(result)

            prediction = model.predict(result)[0]

            print("\nPrediction:",
                  "Positive" if prediction == 1 else "Negative")

check_button = widgets.Button(
    description="Validate and Predict",
    button_style="success"
)

check_button.on_click(check_inputs)

display(check_button, output)

Button(button_style='success', description='Validate and Predict', style=ButtonStyle())

Output()

In [8]:
print("Validation Framework Summary")

print("Total Features:", len(features))
print("Missing Values: Supported")
print("Incorrect Data Types: Supported")
print("Infinite Values: Rejected")
print("Out-of-Range Values: Rejected")
print("Contextual Error Messages: Supported")
print("Input Suggestions: Supported")

Validation Framework Summary
Total Features: 5
Missing Values: Supported
Incorrect Data Types: Supported
Infinite Values: Rejected
Out-of-Range Values: Rejected
Contextual Error Messages: Supported
Input Suggestions: Supported
